* Notebook [riksdagen_64_wikidata.ipynb](https://github.com/salgo60/SCB-Wikidata/blob/main/notebook/riksdagen_64_wikidata.ipynb) 
* [#64](https://github.com/salgo60/SCB-Wikidata/issues/64)
* result   
   * [html](https://salgo60.github.io/SCB-Wikidata/notebook/resultsRiksdagenWD/links_RiksdagenWikidata_v1_2026_02_08.html) 
   * [csv](https://salgo60.github.io/SCB-Wikidata/notebook/resultsRiksdagenWD/links_RiksdagenWikidata_v1_2026_02_08.csv)
 


In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-02-12 10:20:26


In [2]:
WikidataInclude = False
SCRIPT_NAME = "riksdagen_64_wikidata.ipynb"
SCRIPT_URL = (
    "https://github.com/salgo60/SCB-Wikidata/"
    "blob/master/notebook/riksdagen_64_wikidata.ipynb"
) 


In [3]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [4]:
import requests

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; URL-checker/1.0) salgo60@msn.com"
})


In [5]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [6]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []

    # --- language-specific sites ---
    for key, lang_block in data.items():
        if not key.isdigit():
            continue  # skip "count", "specials"

        lang_code = lang_block.get("code")
        lang_name = lang_block.get("name")

        for site in lang_block.get("site", []):
            rows.append({
                "lang_code": lang_code,
                "lang_name": lang_name,
                "project": site.get("project"),
                "url": site.get("url"),
                "dbname": site.get("dbname"),
                "site_name": site.get("sitename"),
                "closed": site.get("closed", False)
            })

    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [7]:
import requests
import pandas as pd


HEADERS = {
    "User-Agent": "salgo60-language-fetcher/2.0 (https://github.com/salgo60) salgo60@msn.com"
}


df_lang_fetch = fetch_sitematrix_df()
df_lang_fetch["closed"] = df_lang_fetch["closed"].fillna(False).astype(bool)

df_lang_wikipedia = df_lang_fetch[
    (df_lang_fetch["site_name"] == "Wikipedia") &
    (
        (df_lang_fetch["lang_name"].str.lower() != "special") |
        (df_lang_fetch["dbname"] == "wikidatawiki")
    )
]  

#df_lang_wikipedia.to_csv("test.csv")
df_lang_wikipedia.info()

[DEBUG] Fetching sitematrix…
<class 'pandas.core.frame.DataFrame'>
Index: 187 entries, 0 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  187 non-null    object
 1   lang_name  186 non-null    object
 2   project    0 non-null      object
 3   url        187 non-null    object
 4   dbname     187 non-null    object
 5   site_name  187 non-null    object
 6   closed     187 non-null    bool  
dtypes: bool(1), object(6)
memory usage: 10.4+ KB


In [8]:
df_lang_wikipedia

,lang_code,lang_name,project,url,dbname,site_name,closed
0,aa,Qafár af,None,https://aa.wikipedia.org,aawiki,Wikipedia,False
5,ace,Acèh,None,https://ace.wikipedia.org,acewiki,Wikipedia,False
7,af,Afrikaans,None,https://af.wikipedia.org,afwiki,Wikipedia,False
11,ak,None,None,https://ak.wikipedia.org,akwiki,Wikipedia,False
18,ami,Pangcah,None,https://ami.wikipedia.org,amiwiki,Wikipedia,False
...,...,...,...,...,...,...,...
924,za,Vahcuengh,None,https://za.wikipedia.org,zawiki,Wikipedia,False
928,zea,Zeêuws,None,https://zea.wikipedia.org,zeawiki,Wikipedia,False
931,zh,中文,None,https://zh.wikipedia.org,zhwiki,Wikipedia,False
939,zu,isiZulu,None,https://zu.wikipedia.org,zuwiki,Wikipedia,False


### Bara köra Wikidata 

In [9]:
df_wd = df_lang_wikipedia[df_lang_wikipedia["lang_code"] == "special"]

In [10]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [11]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [12]:
import os
import time
import random
import requests
import pandas as pd
from urllib.parse import urlparse
from tqdm.notebook import tqdm
file_path_domain = "sources/domains_riksdagen.csv"
domains = read_domains(file_path_domain)
print(domains)


[DEBUG] Reading domains from: sources/domains_riksdagen.csv
[DEBUG] Found 1 domains.
['riksdagen.se']


In [13]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []


    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        print(site)
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [14]:

df_wd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 1047 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  1 non-null      object
 1   lang_name  1 non-null      object
 2   project    0 non-null      object
 3   url        1 non-null      object
 4   dbname     1 non-null      object
 5   site_name  1 non-null      object
 6   closed     1 non-null      bool  
dtypes: bool(1), object(6)
memory usage: 57.0+ bytes


In [15]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [16]:
def resolve_api_base(lang):
    if lang == "special":
        # Wikidata (only valid "special" case in your pipeline)
        return "https://www.wikidata.org/w/api.php"

    return f"https://{lang}.wikipedia.org/w/api.php"


In [17]:
def request_timeout(lang):
    return 100 if lang == "special" else 10

In [18]:
import os

CHECKPOINT_FILE = "checkpoint_riksdagen_wd_links.parquet"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        df_cp = pd.read_parquet(CHECKPOINT_FILE)
        print(f"Checkpoint hittad: {len(df_cp)} rader")
        return df_cp
    return pd.DataFrame()


In [19]:
# -----------------------------------------------------------
# Fetch exturlusage entries for one lang/domain
# -----------------------------------------------------------
def fetch_exturlusage(lang, domain):
    #base = f"https://{lang}.wikipedia.org/w/api.php"
    base = resolve_api_base(lang)
    params = {
        "action": "query",
        "format": "json",
        "list": "exturlusage",
        "euquery": domain,
        "eulimit": "max"
    }
    while True:
        #r = session.get(base, params=params, timeout=10)
        r = session.get(base, params=params, timeout=request_timeout(lang))        
        try:
            data = r.json()
        except ValueError:
            print(f"[WARN] {lang}: JSON decode failed")
            break

        for item in data.get("query", {}).get("exturlusage", []):
            yield {
                "lang": lang,
                "page_title": item.get("title"),
                "url": item.get("url"),
                "wiki_link": f"https://{lang}.wikipedia.org/wiki/{item.get('title').replace(' ', '_')}"
            }

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.3)

In [20]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [21]:
domains

['riksdagen.se']

In [22]:
UseWDCache = True  # True = använd cache, False = hämta från WD
CACHE_FILE = "riksdagen_wd_cache.json"

In [23]:
import json
import os

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}


In [24]:
def fetch_with_cache(lang, domains):
    global cache

    key = f"{lang}|{'|'.join(domains)}"

    # --- Läs från cache ---
    if UseWDCache:
        if key in cache:
            print("CACHE HIT:", key)
            return cache[key]
        else:
            print("CACHE MISS (UseWDCache=True):", key)
            return []

    # --- Hämta från WD ---
    data = list(fetch_exturlusage(lang, domains))

    cache[key] = data

    # skriv till fil direkt (säkert vid långa körningar)
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f)

    return data


In [25]:

# -------------------------
# Session & helpers
# -------------------------
session = requests.Session()
session.headers.update({"User-Agent": "SCB-LinkAudit/1.0 salgo60@msn.com"})


print("Antal Språk:",len(df_wd))

results = []



results = []

for _, row in df_wd.iterrows():
    lang = row["lang_code"]
    url  = row["url"]
    lang_name = row["lang_name"]

    before = len(results)

    print(lang, url, lang_name, domains)

    entries = fetch_with_cache(lang, domains)
    results.extend(entries)

    after = len(results)
    print(lang, url, lang_name, "-", after-before)


Antal Språk: 1
special https://www.wikidata.org special ['riksdagen.se']
CACHE HIT: special|riksdagen.se
special https://www.wikidata.org special - 617191


In [26]:
df_riksdagen_wd = pd.DataFrame(results)
df_riksdagen_wd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617191 entries, 0 to 617190
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   lang        617191 non-null  object
 1   page_title  617191 non-null  object
 2   url         617191 non-null  object
 3   wiki_link   617191 non-null  object
dtypes: object(4)
memory usage: 18.8+ MB


In [27]:
import pandas as pd

# --- Stats ---
total_links = len(df_riksdagen_wd)
total_unique_links = df_riksdagen_wd['url'].nunique()
num_languages = df_riksdagen_wd['lang'].nunique()
langs_sorted = df_riksdagen_wd['lang'].value_counts()

print("Total links:", total_links)
print("Total unique links:", total_unique_links)
print("Wikidata")


Total links: 617191
Total unique links: 614687
Wikidata


### Check url

In [28]:
import requests 
from requests.exceptions import RequestException 
from concurrent.futures import ThreadPoolExecutor, as_completed 
from tqdm import tqdm 
import pandas as pd 
import time  
import threading

In [29]:
MAX_WORKERS = 64              # För IO-bound scraping kan du ofta köra 32–128 workers
MAX_WORKERS = min(128, os.cpu_count()*10)
REQUEST_TIMEOUT = 15

RATE_LIMIT_PER_SEC = 2       # max requests/sek totalt
BACKOFF_FACTOR = 2
MAX_RETRIES = 3

CHECKPOINT_EVERY = 500
CHUNK_SIZE = 5000

CHECKPOINT_FILE = "checkpoint_riksdagen_wikidata.parquet"


In [30]:
import threading
import time

rate_lock = threading.Lock()
last_request_time = 0

def wait_for_rate_limit():
    global last_request_time
    
    with rate_lock:
        now = time.time()
        wait = 1 / RATE_LIMIT_PER_SEC - (now - last_request_time)
        if wait > 0:
            time.sleep(wait)
        last_request_time = time.time()


In [31]:
def request_with_retry(method, url, session, **kwargs):
    delay = 1
    
    for attempt in range(MAX_RETRIES):
        try:
            wait_for_rate_limit()
            return session.request(method, url, **kwargs)
        
        except RequestException:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(delay)
            delay *= BACKOFF_FACTOR


In [32]:
def check_url(url: str) -> dict:
    session = get_session()

    try:
        r = request_with_retry(
            "HEAD",
            url,
            session,
            allow_redirects=False,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {"url": url, "status": "error", "reason": str(e)}

    final_url = r.url

    if r.status_code >= 400:
        return {"url": url, "status": "dead", "reason": f"HTTP {r.status_code}"}

    if norm(final_url) == norm(ROOT_CANONICAL) and norm(url) != norm(final_url):
        return {"url": url, "status": "dead", "reason": "redirect_to_root"}

    try:
        r = request_with_retry(
            "GET",
            final_url,
            session,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {"url": url, "status": "error", "reason": str(e)}

    if looks_like_soft_404(r):
        return {"url": url, "status": "dead", "reason": "soft_404"}

    return {"url": url, "status": "ok"}
    

In [33]:
import os

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        return pd.read_parquet(CHECKPOINT_FILE)
    return pd.DataFrame()


In [34]:
from itertools import islice

def chunks(iterable, size):
    it = iter(iterable)
    while chunk := list(islice(it, size)):
        yield chunk


In [35]:
# ========================== # Internet Archive # ========================== 
def check_internet_archive(url: str) -> str | None: 
    session = get_session() 
    api = "https://archive.org/wayback/available" 
    
    try: 
        r = session.get( api, 
                        params={"url": url}, 
                        timeout=10, ) 
        data = r.json() 
    except Exception: 
        return None 
    snap = data.get("archived_snapshots", {}).get("closest") 
    if snap and snap.get("available"): 
        return snap.get("url") 
    return None

In [36]:


def is_worth_checking(url: str) -> bool: 
    if not url.startswith("http"):
        return False 
    if "riksdagen.se" not in url: 
        return False 
    return True

In [37]:
# ========================== # Worker # ========================== 
def process_url(url: str) -> dict: 
    result = check_url(url) 
    if result["status"] == "dead": 
        ia_url = check_internet_archive(url) 
        result["ia_url"] = ia_url 
        result["ia_status"] = "available" if ia_url else "missing" 
    else: 
        result["ia_url"] = None 
        result["ia_status"] = "skipped" 
    return result

In [38]:
def run(df):

    urls_all = [
        u for u in df["url"].dropna().astype(str).unique()
        if is_worth_checking(u)
    ]

    cp = load_checkpoint()
    done = set(cp["url"]) if not cp.empty else set()

    urls_remaining = [u for u in urls_all if u not in done]

    print("Totalt:", len(urls_all))
    print("Redan klara:", len(done))
    print("Kvar:", len(urls_remaining))
    print("MAX_WORKERS:", MAX_WORKERS)

    results = cp.to_dict("records") if not cp.empty else []

    processed_since_log = 0

    for chunk_idx, chunk in enumerate(chunks(urls_remaining, CHUNK_SIZE), 1):

        print(f"\nStartar chunk {chunk_idx} ({len(chunk)} URL:er)")

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process_url, u) for u in chunk]

            from tqdm import tqdm

            for chunk_idx, chunk in enumerate(chunks(urls_remaining, CHUNK_SIZE), 1):
            
                print(f"\nStartar chunk {chunk_idx} ({len(chunk)} URL:er)")
            
                with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                    futures = [ex.submit(process_url, u) for u in chunk]
            
                    for f in tqdm(
                        as_completed(futures),
                        total=len(futures),
                        desc=f"Chunk {chunk_idx}",
                    ):
                        res = f.result()
                        results.append(res)
            
                        processed_since_log += 1
            
                        # --- Logga var 500:e ---
                        if processed_since_log >= 500:
                            print(f"{len(results)} totalt klara...")
                            processed_since_log = 0
            
                        # --- Checkpoint ---
                        if len(results) % CHECKPOINT_EVERY == 0:
                            tmp = CHECKPOINT_FILE + ".tmp"
                            pd.DataFrame(results).to_parquet(tmp)
                            os.replace(tmp, CHECKPOINT_FILE)
                            print(f"Checkpoint sparad ({len(results)})")

    df_final = pd.DataFrame(results)
    df_final.to_parquet("final.parquet")

    print("\nKLAR.")
    print("Totalt processade:", len(df_final))

    return df_final


In [39]:

import requests
import threading

USER_AGENT = "LinkChecker/1.0 salgo60@msn.com"

thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        thread_local.session = s
    return thread_local.session


def norm(u: str) -> str:
    return u.rstrip("/").lower()


def is_worth_checking(url: str) -> bool:
    return url.startswith("http") and "riksdagen.se" in url



In [40]:
def looks_like_soft_404(response) -> bool:
    text = (response.text or "").lower()

    for phrase in SOFT_404_PHRASES:
        if phrase in text:
            return True

    if "<title>" in text:
        title = text.split("<title>", 1)[1].split("</title>", 1)[0]
        if "404" in title.lower():
            return True
        if "sidan finns inte" in title.lower():
            return True

    return False

SOFT_404_PHRASES = [
    "sidan kan inte hittas",
    "sidan tagits bort",
    "felaktig adress",
    "kontakta registrator",
    "sidan finns inte",
    "den här sidan kan inte visas",
    "Hoppsan! Vi kunde tyvärr inte hitta sidan"
]

USER_AGENT = "LinkChecker/1.0 (research; salgo60@msn.com)"


In [41]:
ROOT_CANONICAL = "https://www.riksdagen.se"

In [ ]:
df_result = run(df_riksdagen_wd)

from datetime import date
import os

# Sätt datum
today = date.today().strftime("%Y_%m_%d")

# Se till att katalogen finns
os.makedirs("resultsRiksdagenWD", exist_ok=True)

# Bygg filnamn
outfile = f"resultsRiksdagenWD/links_Riksdagen_v1_Wikidata_{today}.csv"
    
# Exportera
df_riksdagen_wd.to_csv(outfile, index=False, encoding="utf-8")

print(f"[OK] Exported {len(df_riksdagen_wd)} rows to {outfile}")


Totalt: 614687
Redan klara: 134000
Kvar: 480687
MAX_WORKERS: 80

Startar chunk 1 (5000 URL:er)

Startar chunk 1 (5000 URL:er)


Chunk 1:  10%|███                            | 500/5000 [18:18<45:19,  1.65it/s]

134500 totalt klara...
Checkpoint sparad (134500)


Chunk 1:  20%|█████▊                       | 999/5000 [51:01<1:04:38,  1.03it/s]

135000 totalt klara...


...

Chunk 35:  20%|█████▏                    | 999/5000 [1:02:13<7:25:52,  6.69s/it]

305000 totalt klara...


Chunk 35:  20%|████▊                   | 1000/5000 [1:05:02<61:55:57, 55.74s/it]

Checkpoint sparad (305000)


Chunk 35:  30%|███████▍                 | 1486/5000 [1:46:25<4:33:06,  4.66s/it]

In [ ]:
lang_stats = (
    df_riksdagen_wd
    .groupby("lang")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        archived_links=("ia_status", lambda s: (s == "available").sum()),
    )
    .reset_index()
)

lang_stats["broken_pct"] = (
    100 * lang_stats["broken_links"] / lang_stats["total_links"]
).round(1)

lang_stats["broken_lost"] = (
    lang_stats["broken_links"] - lang_stats["archived_links"])

top10_langs = (
    lang_stats
    .sort_values("total_links", ascending=False)
    .head(10)
)
top10_langs[
    [
        "lang",
        "total_links",
        "broken_links",
        "broken_pct",
        "archived_links",
        "broken_lost",
    ]
]

In [ ]:
from urllib.parse import urlparse

df = df_riksdagen_wd.copy()

df["domain"] = df["url"].apply(
    lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
)
domain_stats = (
    df
    .groupby("domain")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        error_links=("status", lambda s: (s == "error").sum()),
    )
    .reset_index()
)
domain_stats["broken_pct"] = (
    100 * domain_stats["broken_links"] / domain_stats["total_links"]
).round(1)

domain_stats["error_pct"] = (
    100 * domain_stats["error_links"] / domain_stats["total_links"]
).round(1)


In [ ]:
status_counts = df["status"].value_counts()

num_ok = int(status_counts.get("ok", 0))
num_dead = int(status_counts.get("dead", 0))
num_error = int(status_counts.get("error", 0))
num_total = len(df)

pct_ok = round(100 * num_ok / num_total, 1)
pct_dead = round(100 * num_dead / num_total, 1)
pct_error = round(100 * num_error / num_total, 1)

# Broken links: archived vs lost
num_dead_archived = df[
    (df["status"] == "dead") & (df["ia_status"] == "available")
].shape[0]

num_dead_lost = num_dead - num_dead_archived


In [ ]:
top_domains = (
    domain_stats[domain_stats["total_links"] >= 5]
    .sort_values("total_links", ascending=False)
    .head(10)
)
domain_stats_html = "<ul>"
for _, r in top_domains.iterrows():
    domain_stats_html += (
        f"<li><strong>{r['domain']}</strong>: "
        f"{r['broken_links']} / {r['total_links']} broken "
        f"({r['broken_pct']}%)</li>"
    )
domain_stats_html += "</ul>"


In [ ]:
from pathlib import Path
from datetime import date, datetime
from urllib.parse import quote
import pandas as pd


def save_sortable_html_df_riksdagen_wd(
    df,
    out_dir="resultsRiksdagenWD",
    domains=None,
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/64",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    out_path = out_dir / f"links_riksdagen_v1_Wikidata_{today}.html"
 
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

     # --- Förbered data ---
    df = df.copy()  
    status_counts = df["status"].value_counts()

    num_ok = int(status_counts.get("ok", 0))
    num_dead = int(status_counts.get("dead", 0))
    num_error = int(status_counts.get("error", 0))
    num_total = len(df)
    
    pct_ok = round(100 * num_ok / num_total, 1)
    pct_dead = round(100 * num_dead / num_total, 1)
    pct_error = round(100 * num_error / num_total, 1)
    
    # Broken links: archived vs lost
    num_dead_archived = df[
        (df["status"] == "dead") & (df["ia_status"] == "available")
    ].shape[0]
    
    num_dead_lost = num_dead - num_dead_archived

    domains = domains or []
    from urllib.parse import urlparse
    
    df["domain"] = df["url"].apply(
        lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
    )

    domain_stats = (
        df
        .groupby("domain")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
        )
        .reset_index()
    )
    
    domain_stats["broken_pct"] = (
        100 * domain_stats["broken_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["error_pct"] = (
        100 * domain_stats["error_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["problem_pct"] = (
        100 * (domain_stats["broken_links"] + domain_stats["error_links"])
        / domain_stats["total_links"]
    ).round(1)
    

    domain_stats_html = "<ul>"
    for _, r in top_domains.iterrows():
        domain_stats_html += (
            f"<li><strong>{r['domain']}</strong>: "
            f"{r['broken_links']} / {r['total_links']} broken "
            f"({r['broken_pct']}%)</li>"
        )
    domain_stats_html += "</ul>"

    domain_table_html = (
        domain_stats
        .head(20)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )
    lang_stats = (
    df
        .groupby("lang")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
            broken_archived=("ia_status", lambda s: (s == "available").sum()),
        )
        .reset_index()
    )
    
    lang_stats["broken_lost"] = (
        lang_stats["broken_links"] - lang_stats["broken_archived"]
    )
    
    lang_stats["broken_pct"] = (
        100 * lang_stats["broken_links"] / lang_stats["total_links"]
    ).round(1)
    
    lang_stats["problem_pct"] = (
        100 * (lang_stats["broken_links"] + lang_stats["error_links"])
        / lang_stats["total_links"]
    ).round(1)

    lang_stats = lang_stats.sort_values(
        "broken_links",
        ascending=False
    )
 
    lang_stats_display = lang_stats[
        [
            "lang",
            "total_links",
            "broken_links",
            "broken_archived",
            "broken_lost",
            "broken_pct",
            "problem_pct",
        ]
    ]

    lang_table_html = (
        lang_stats_display
        .head(15)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )

    STATUS_ICON = {
        "ok":    ("fa-circle-check", "#2e7d32", "OK"),
        "dead":  ("fa-circle-xmark", "#c62828", "Broken link"),
        "error": ("fa-triangle-exclamation", "#ef6c00", "Request error"),
    }
    
    if "status" in df.columns:
        def render_status(r):
            icon, color, label = STATUS_ICON.get(
                r["status"], ("fa-question-circle", "#757575", "Unknown")
            )
            reason = r.get("reason", "")
            return (
                f'<span class="status-icon" '
                f'data-status="{r["status"]}" '
                f'title="{label}: {reason}" '
                f'style="color:{color}; font-size:14px; cursor:pointer;">'
                f'<i class="fa-solid {icon}"></i>'
                f'</span>'
            )
        df.insert(
            0,
            "status_icon",
            df.apply(render_status, axis=1)
        )

        def render_ia_icon(r):
            if r.get("ia_status") == "available" and r.get("ia_url"):
                return (
                    f'<a href="{r["ia_url"]}" target="_blank" '
                    f'title="Archived copy (Internet Archive)">'
                    f'<i class="fa-solid fa-box-archive" '
                    f'style="color:#1565c0;"></i>'
                    f'</a>'
                )
            return ""
        
        df.insert(
            1,
            "archive",
            df.apply(render_ia_icon, axis=1)
        )


    # Wikipedia: ikon + titel (byggd från lang + page_title)
    if {"lang", "page_title"}.issubset(df.columns):
        df["page_title"] = df.apply(
            lambda r: (
                f'<a href="https://{r["lang"]}.wikipedia.org/wiki/{quote(str(r["page_title"]))}" '
                f'target="_blank" title="Wikipedia ({r["lang"]})">'
                f'<i class="fa-brands fa-wikipedia-w" style="margin-right:6px;"></i>'
                f'{r["page_title"]}</a>'
                if pd.notna(r["lang"]) and pd.notna(r["page_title"])
                else r["page_title"]
            ),
            axis=1,
        )

    # Externa länkar
    for col in ["Wikipedia-länk", "Extern länk", "url"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if pd.notna(x) else ""
            )

    # --- HTML-tabell ---
    html_table = df.to_html(
        classes="pivot",
        border=0,
        escape=False,  # krävs för HTML-länkar
        index=False,
    )

    # --- CSS ---
    css = """
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 20px;
      }
      table.pivot {
        border-collapse: collapse;
        width: 100%;
        font-size: 12px;
      }
      table.pivot th, table.pivot td {
        border: 1px solid #999;
        padding: 6px 8px;
        text-align: left;
        vertical-align: top;
        white-space: normal;
      }
      table.pivot th {
        cursor: pointer;
        background: #f2f2f2;
      }
      table.pivot th:hover {
        background: #e2e2e2;
      }
      table.pivot thead th {
        position: sticky;
        top: 0;
        background: #f2f2f2;
        z-index: 2;
      }
      table.pivot th::after {
        content: "";
        float: right;
        opacity: 0.4;
      }
      table.pivot th.sorted-asc::after {
        content: " ▲";
      }
      table.pivot th.sorted-desc::after {
        content: " ▼";
      }
      /* Row coloring by status */
      table.pivot tr[data-status="dead"] {
         background-color: #fdecea;  /* light red */
      }
      table.pivot tr[data-status="dead"] td:nth-child(2) i {
          color: #1565c0;
        }

      table.pivot tr[data-status="error"] {
          background-color: #fff4e5;  /* light orange */
      }

      table.pivot td a {
        color: #0645ad;
        text-decoration: none;
      }
      table.pivot td a:hover {
        text-decoration: underline;
      }
      .meta {
        background: #f8f8f8;
        border: 1px solid #ccc;
        padding: 12px;
        margin-bottom: 20px;
        font-size: 13px;
      }
      .meta h2 {
        margin-top: 0;
      }
    </style>
    """

    # --- JavaScript (sortering) ---
    js = """
    <script>
    document.addEventListener('DOMContentLoaded', () => {
        // Propagate status from first cell to row
        document.querySelectorAll("table.pivot tbody tr").forEach(row => {
            const statusCell = row.querySelector(".status-icon");
            if (statusCell) {
                row.dataset.status = statusCell.dataset.status;
            }
        });
        let showOnlyBroken = false;

        document.querySelectorAll(".status-icon").forEach(icon => {
            icon.addEventListener("click", event => {
                event.stopPropagation(); // prevent column sort
                showOnlyBroken = !showOnlyBroken;
        
                document.querySelectorAll("table.pivot tbody tr").forEach(row => {
                    if (showOnlyBroken) {
                        row.style.display =
                            row.dataset.status === "dead" ? "" : "none";
                    } else {
                        row.style.display = "";
                    }
                });
            });
        });

        document.querySelectorAll("table.pivot th").forEach((header, colIndex) => {
            header.addEventListener("click", () => {
                const table = header.closest("table");
                const tbody = table.querySelector("tbody");
                const rows = Array.from(tbody.querySelectorAll("tr"));
                const asc = !header.classList.contains("sorted-asc");

                rows.sort((a, b) => {
                    const A = a.children[colIndex].innerText.trim();
                    const B = b.children[colIndex].innerText.trim();
                    const numA = parseFloat(A.replace(",", "."));
                    const numB = parseFloat(B.replace(",", "."));
                    if (!isNaN(numA) && !isNaN(numB)) {
                        return asc ? numA - numB : numB - numA;
                    }
                    return asc ? A.localeCompare(B) : B.localeCompare(A);
                });

                table.querySelectorAll("th").forEach(th =>
                    th.classList.remove("sorted-asc", "sorted-desc")
                );
                header.classList.add(asc ? "sorted-asc" : "sorted-desc");
                rows.forEach(row => tbody.appendChild(row));
            });
        });
    });
    </script>
    """
    status_counts = df["status"].value_counts()
    
    num_ok = status_counts.get("ok", 0)
    num_dead = status_counts.get("dead", 0)
    num_error = status_counts.get("error", 0)
    num_total = len(df)

    # --- Metadata ---
    meta_html = f"""
    <div class="meta">
      <h2>Summary</h2>
    
      <p><strong>Rerun:</strong> {rerun_ts}</p>
      <p><strong>Script:</strong>
         <a href="{SCRIPT_URL}" target="_blank">{SCRIPT_NAME}</a>
      </p>
    
      <p>
        <strong>Links checked:</strong> {num_total}<br>
        <strong style="color:#2e7d32;">OK:</strong> {num_ok} ({pct_ok}%)<br>
        <strong style="color:#c62828;">Broken:</strong> {num_dead} ({pct_dead}%)<br>
        &nbsp;&nbsp;↳ Archived: {num_dead_archived}<br>
        &nbsp;&nbsp;↳ Lost: {num_dead_lost}<br>
        <strong style="color:#ef6c00;">Errors:</strong> {num_error} ({pct_error}%)
      </p>
      <p><strong>Issue:</strong>
         <a href="{issue_url}" target="_blank">{issue_url.split("/")[-1]}</a>
      </p>
    
      <p><strong>Datakällor:</strong><br>
         Wikidata<br>
         MediaWiki API – exturlusage
      </p>
    
      <h2>Domains with broken links</h2>
      <p>Top domains ranked by broken-link impact.</p>
      {domain_table_html}
    </div>
    """


    # --- Slutlig HTML ---
    title_text = "riksdagen.se links in Wikidata" 
    if WikidataInclude: 
        title_text = " and Wikidata"
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>{title_text}/title>
      <link rel="stylesheet"
            href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.1/css/all.min.css">
      {css}
    </head>
    <body>
      <h1>Wikidata → Riksdagen v1</h1>
      {meta_html}
      <p>Sorterbar tabell. Klicka på kolumnrubriker för sortering.</p>
      {html_table}
      {js}
      <h2>Broken-link summary by Wikipedia language</h2>
     <p>
       Languages ranked by broken-link impact (broken + error links).
     </p>
    {lang_table_html}
    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ HTML skapad: {out_path}")


In [ ]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))
